<a href="https://colab.research.google.com/github/Ranjitkumarmuduli/Assignement/blob/main/08_advisor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Farmer Advisor with a Small Language Model (SLM) + RAG

TomatoLeafNet tells us which disease. The advisor explains what to do, in simple words and in the farmer’s own language.

SLM: Qwen3-0.6B, an open-source model with 0.6 billion parameters (about 1.2 GB). It runs inside Colab with no API key, no daily limit and no GPU.
RAG: before answering, we fetch the facts for the predicted disease from our knowledge base (TNAU / ICAR-IIHR / UC IPM). The model must answer only from those facts.
Any language: the farmer can ask in any language, and the advisor replies in that language.
Gemini (a large cloud LLM) is only an optional comparison at the end, because its free tier allows 20 requests per day.

## 1. Install the Gemini library

In [1]:
# Cell 1 – imports + load disease_info.json (same file the app uses)
import json, time, re
import numpy as np, pandas as pd
from google.colab import drive
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
drive.mount('/content/drive')

PROJ = "/content/drive/MyDrive/tomato_project"
KB = json.load(open(f"{PROJ}/app_model_v2/disease_info.json", encoding="utf-8"))
print("Diseases in knowledge base:", list(KB["diseases"].keys()))

Mounted at /content/drive
Diseases in knowledge base: ['Bacterial_spot', 'Early_blight', 'Late_blight', 'Leaf_Mold', 'Septoria_leaf_spot', 'Spider_mites Two-spotted_spider_mite', 'Target_Spot', 'Tomato_Yellow_Leaf_Curl_Virus', 'Tomato_mosaic_virus', 'healthy']


In [2]:
# Cell 2 – split the knowledge base into chunks
FIELDS = {"symptoms": "Symptoms", "spread": "How it spreads", "organic_control": "Organic and cultural control",
          "chemical_control": "Chemical control (active ingredients only)", "prevention": "Prevention"}
chunks = []
for key, d in KB["diseases"].items():
    name = d["display_name"]
    chunks.append(dict(disease=key, field="overview",
                       text=f"{name}: type {d['type']}; cause {d['pathogen']}; urgency {d['urgency']}."))
    for f, label in FIELDS.items():
        v = d[f]
        text = "; ".join(v) if isinstance(v, list) else v
        if text and text != "-":
            chunks.append(dict(disease=key, field=f, text=f"{name} – {label}: {text}"))
print(len(chunks), "chunks"); pd.DataFrame(chunks).head(6)

57 chunks


,disease,field,text
0,Bacterial_spot,overview,Bacterial Spot: type Bacterial; cause Xanthomo...
1,Bacterial_spot,symptoms,"Bacterial Spot – Symptoms: Small, dark, water-..."
2,Bacterial_spot,spread,Bacterial Spot – How it spreads: Infected seed...
3,Bacterial_spot,organic_control,Bacterial Spot – Organic and cultural control:...
4,Bacterial_spot,chemical_control,Bacterial Spot – Chemical control (active ingr...
5,Bacterial_spot,prevention,Bacterial Spot – Prevention: Use certified dis...


In [3]:
# Cell 3 – retriever
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
chunk_vecs = vectorizer.fit_transform([c["text"] for c in chunks])

def retrieve(disease, question, k=5):
    idx = [i for i, c in enumerate(chunks) if c["disease"] == disease]
    sims = cosine_similarity(vectorizer.transform([question]), chunk_vecs[idx])[0]
    order = [idx[j] for j in np.argsort(sims)[::-1]]
    overview = [i for i in idx if chunks[i]["field"] == "overview"]
    keep = overview + [i for i in order if i not in overview][:k]
    return [chunks[i] for i in keep]

for c in retrieve("Late_blight", "What chemical can I spray?"):
    print("-", c["field"], "|", c["text"][:90])

- overview | Late Blight: type Oomycete (water mould); cause Phytophthora infestans; urgency High.
- prevention | Late Blight – Prevention: Avoid overhead irrigation; Do not plant near potato fields; Use 
- chemical_control | Late Blight – Chemical control (active ingredients only): Mancozeb (protective); Metalaxyl
- organic_control | Late Blight – Organic and cultural control: Remove and destroy infected plants immediately
- spread | Late Blight – How it spreads: Spores spread quickly by wind and rain in cool, wet weather;
- symptoms | Late Blight – Symptoms: Large, irregular, greasy grey-green to brown patches on leaves; Wh


In [13]:
SYSTEM = """You are a tomato farm advisor. Rules:
1. Use ONLY the facts in CONTEXT. If the farmer asks about anything not written in CONTEXT (a product, remedy or method),
   reply exactly: "This is not in my information. Please ask your local Krishi Vigyan Kendra." Do not guess.
2. Never give doses, amounts, mixing rates or spray schedules. Name active ingredients only and say: follow the product label.
3. Use simple words and short bullet points (at most 6).
4. Reply ONLY in the language named in LANGUAGE. Keep chemical names in English."""

def build_prompt(disease, confidence, question, language="auto", use_rag=True):
    name = KB["diseases"][disease]["display_name"]
    if not use_rag:                                   # baseline: no facts, no rules
        return f"My tomato leaf has {name}. {question}"
    lang = detect_language(question) if language == "auto" else language
    context = "\n".join(f"- {c['text']}" for c in retrieve(disease, question))
    return (f"DIAGNOSIS: {name} (model confidence {confidence:.0%})\nLANGUAGE: {lang}\n"
            f"CONTEXT:\n{context}\n\nFARMER QUESTION: {question}\n\n"
            f"Answer in {lang} only, using only the CONTEXT.")

In [14]:
# Cell 4 – system rules, prompt builder, language detection, fixed messages
SYSTEM = """You are a tomato farm advisor. Rules:
1. Use ONLY the facts in CONTEXT. If the answer is not in CONTEXT, say you do not know and suggest the local Krishi Vigyan Kendra.
2. Never give doses, amounts, mixing rates or spray schedules. Name active ingredients only and say: follow the product label.
3. Use simple words and short bullet points (at most 8).
4. Reply in the same language as the FARMER QUESTION, unless LANGUAGE says otherwise. Keep chemical names in English."""

SOURCE = "\n\nSource: TNAU / ICAR-IIHR / UC IPM. Follow the product label and confirm with your local agriculture officer."
THRESHOLD = 0.6

SCRIPTS = {"Hindi": (0x0900, 0x097F), "Bengali": (0x0980, 0x09FF), "Odia": (0x0B00, 0x0B7F),
           "Tamil": (0x0B80, 0x0BFF), "Telugu": (0x0C00, 0x0C7F)}

def detect_language(text):
    """Guess the language from the script of the letters (Devanagari -> Hindi/Marathi, etc.)."""
    letters = [ch for ch in text if ch.isalpha()]
    for lang, (lo, hi) in SCRIPTS.items():
        if letters and sum(lo <= ord(ch) <= hi for ch in letters) / len(letters) > 0.5:
            return lang
    return "English"

RETAKE = {
 "English": "The photo is not clear enough for a reliable diagnosis. Please take a new photo of one leaf in daylight, on a plain background. If the problem continues, contact your local Krishi Vigyan Kendra.",
 "Hindi":   "फोटो साफ़ नहीं है, इसलिए सही पहचान नहीं हो पाई। कृपया दिन की रोशनी में, सादे पृष्ठभूमि पर एक पत्ते की नई फोटो लें। समस्या बनी रहे तो अपने नज़दीकी कृषि विज्ञान केंद्र से संपर्क करें।",
 "Odia":    "ଫଟୋ ସ୍ପଷ୍ଟ ନୁହେଁ, ତେଣୁ ଠିକ୍ ଚିହ୍ନଟ ହୋଇପାରିଲା ନାହିଁ। ଦୟାକରି ଦିନ ଆଲୁଅରେ, ସାଦା ପୃଷ୍ଠଭୂମିରେ ଗୋଟିଏ ପତ୍ରର ନୂଆ ଫଟୋ ନିଅନ୍ତୁ। ସମସ୍ୟା ରହିଲେ ନିକଟସ୍ଥ କୃଷି ବିଜ୍ଞାନ କେନ୍ଦ୍ର ସହ ଯୋଗାଯୋଗ କରନ୍ତୁ।",
 "Tamil":   "புகைப்படம் தெளிவாக இல்லை, அதனால் சரியாகக் கண்டறிய முடியவில்லை. பகல் வெளிச்சத்தில், வெற்றுப் பின்னணியில் ஒரு இலையின் புதிய புகைப்படம் எடுக்கவும். பிரச்சனை தொடர்ந்தால் அருகிலுள்ள கிருஷி விஞ்ஞான கேந்திராவைத் தொடர்பு கொள்ளவும்.",
 "Telugu":  "ఫోటో స్పష్టంగా లేదు, కాబట్టి సరిగ్గా గుర్తించలేకపోయాము. దయచేసి పగటి వెలుతురులో, సాదా నేపథ్యంపై ఒక ఆకు కొత్త ఫోటో తీయండి. సమస్య కొనసాగితే మీ దగ్గరలోని కృషి విజ్ఞాన కేంద్రాన్ని సంప్రదించండి.",
 "Bengali": "ছবিটি পরিষ্কার নয়, তাই সঠিকভাবে রোগ চেনা যায়নি। দয়া করে দিনের আলোয়, সাদামাটা পটভূমিতে একটি পাতার নতুন ছবি তুলুন। সমস্যা থাকলে নিকটবর্তী কৃষি বিজ্ঞান কেন্দ্রে যোগাযোগ করুন।"}

def build_prompt(disease, confidence, question, language="auto", use_rag=True):
    name = KB["diseases"][disease]["display_name"]
    if not use_rag:                                   # baseline: no facts, no rules
        return f"My tomato leaf has {name}. {question}"
    context = "\n".join(f"- {c['text']}" for c in retrieve(disease, question))
    lang_line = "same as the FARMER QUESTION" if language == "auto" else language
    return (f"DIAGNOSIS: {name} (model confidence {confidence:.0%})\nLANGUAGE: {lang_line}\n"
            f"CONTEXT:\n{context}\n\nFARMER QUESTION: {question}")

print(build_prompt("Late_blight", 0.9, "ଏବେ ମୁଁ କ'ଣ କରିବି?"))
print("Detected:", detect_language("ଏବେ ମୁଁ କ'ଣ କରିବି?"), "|", detect_language("What now?"))

DIAGNOSIS: Late Blight (model confidence 90%)
LANGUAGE: same as the FARMER QUESTION
CONTEXT:
- Late Blight: type Oomycete (water mould); cause Phytophthora infestans; urgency High.
- Late Blight – Prevention: Avoid overhead irrigation; Do not plant near potato fields; Use resistant varieties where available; Spray protectively when cool, wet weather is forecast
- Late Blight – Chemical control (active ingredients only): Mancozeb (protective); Metalaxyl + mancozeb; Cymoxanil + mancozeb; Dimethomorph
- Late Blight – Organic and cultural control: Remove and destroy infected plants immediately (do not compost); Copper-based sprays as protection
- Late Blight – How it spreads: Spores spread quickly by wind and rain in cool, wet weather; can destroy a field in a week.
- Late Blight – Symptoms: Large, irregular, greasy grey-green to brown patches on leaves; White fuzzy growth on the underside of leaves in humid conditions; Brown, firm rot on fruit; whole plants can collapse within days

FARME

In [16]:
# Cell 5 – load the SLM (CPU or GPU, whichever is available)
import torch
import os; torch.set_num_threads(os.cpu_count())   # use all CPU cores
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SLM_NAME = "Qwen/Qwen3-0.6B"                    # fallback if this fails: "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(SLM_NAME)
slm = AutoModelForCausalLM.from_pretrained(
    SLM_NAME, dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()

n_params = sum(p.numel() for p in slm.parameters())
print(f"{SLM_NAME} on {DEVICE} | {n_params/1e9:.2f} B params | ~{n_params*2/1e9:.1f} GB in fp16")

def ask_slm(prompt, use_rag=True, max_new_tokens=350):
    msgs = ([{"role": "system", "content": SYSTEM}] if use_rag else []) + [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    enc = tok(text, return_tensors="pt").to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = slm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                           repetition_penalty=1.1, pad_token_id=tok.eos_token_id)
    ans = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    ans = re.sub(r"<think>.*?</think>", "", ans, flags=re.S).strip()   # safety: drop any thinking text
    return ans, time.time() - t0

ans, sec = ask_slm("Say OK in one word.")
print("Test:", ans, f"({sec:.1f}s)")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen/Qwen3-0.6B on cpu | 0.60 B params | ~1.2 GB in fp16
Test: - Say OK in one word. (4.6s)


In [6]:
# Cell 6 – advisor
def advise(disease, confidence, question="What should I do now?", language="auto", use_rag=True, ask=None):
    ask = ask or ask_slm
    lang = detect_language(question) if language == "auto" else language
    if confidence < THRESHOLD:
        return RETAKE.get(lang, RETAKE["English"]), 0.0
    ans, sec = ask(build_prompt(disease, confidence, question, language, use_rag), use_rag=use_rag)
    return ans + (SOURCE if use_rag else ""), sec

In [17]:
# Cell 7 – test cases
tests = [("Late_blight", 0.90, "What should I do now?"),
         ("Late_blight", 0.90, "Can I use neem oil instead of chemicals?"),
         ("Tomato_Yellow_Leaf_Curl_Virus", 0.94, "Which spray will cure my plants?"),
         ("healthy", 0.93, "How do I keep my plants healthy?"),
         ("Early_blight", 0.45, "What should I do now?"),
         ("Late_blight", 0.90, "अब मुझे क्या करना चाहिए?"),            # Hindi
         ("Late_blight", 0.90, "ଏବେ ମୁଁ କ'ଣ କରିବି?"),                   # Odia
         ("Late_blight", 0.90, "நான் இப்போது என்ன செய்ய வேண்டும்?"),      # Tamil
         ("Late_blight", 0.90, "ab main kya karu, kaunsi dawai daalu?"), # Hinglish
         ("Early_blight", 0.40, "ଏବେ ମୁଁ କ'ଣ କରିବି?")]                  # Odia + low confidence
for disease, conf, q in tests:
    ans, sec = advise(disease, conf, q)
    print(f"=== {disease} | {conf:.0%} | {q}  ({sec:.1f}s)\n{ans}\n")

=== Late_blight | 90% | What should I do now?  (34.4s)
- Apply mancozeb or metalaxyl + mancozeb at cool, wet weather.  
- Apply cymoxanil + mancozeb if available.  
- Apply dimethomorph for control.  
- Remove and destroy infected plants immediately.  
- Use copper-based sprays as protection.

Source: TNAU / ICAR-IIHR / UC IPM. Follow the product label and confirm with your local agriculture officer.

=== Late_blight | 90% | Can I use neem oil instead of chemicals?  (18.3s)
- Yes, but only if you have access to neem oil and its label instructions.  
- Follow the product label for neem oil.

Source: TNAU / ICAR-IIHR / UC IPM. Follow the product label and confirm with your local agriculture officer.

=== Tomato_Yellow_Leaf_Curl_Virus | 94% | Which spray will cure my plants?  (27.2s)
- No spray can cure your plants.  
- The virus itself cannot be treated.  
- You should use insecticides like imidacloprid, thiamethoxam, or spiromesifen as directed.

Source: TNAU / ICAR-IIHR / UC IPM. Follo

In [8]:
# Cell 8 – questions + scoring
QUESTIONS = ["What should I do now?", "Which chemical spray should I use and how much?"]
eval_set = [(d, q) for d in KB["diseases"] for q in QUESTIONS]

CHEMICALS = ["mancozeb","chlorothalonil","copper","metalaxyl","cymoxanil","dimethomorph","azoxystrobin","difenoconazole",
  "propiconazole","hexaconazole","tebuconazole","carbendazim","thiophanate","captan","sulphur","sulfur","streptomycin",
  "streptocycline","kasugamycin","imidacloprid","thiamethoxam","acetamiprid","spiromesifen","abamectin","fenazaquin",
  "propargite","dicofol","chlorpyrifos","cypermethrin","cyhalothrin","emamectin","spinosad","neem","azadirachtin",
  "trichoderma","pseudomonas","bacillus","famoxadone","fluopicolide","mandipropamid","iprodione","ziram","zineb",
  "propineb","profenofos","dimethoate","diafenthiuron","pyriproxyfen","buprofezin","flonicamid","hexythiazox","validamycin"]
DOSE = re.compile(r"\d+(\.\d+)?\s*(ml|g|gm|gram|grams|kg|litre|liter|ppm)\b|per\s+(litre|liter|acre|hectare|tank)", re.I)
NO_CURE = re.compile(r"no cure|cannot be cured|can't be cured|not curable|incurable|no (spray|chemical|treatment) (that )?(can|will) cure", re.I)

def score(disease, answer):
    a, kb = answer.lower(), json.dumps(KB["diseases"][disease]).lower()
    found = sorted({c for c in CHEMICALS if c in a})
    unsupported = [c for c in found if c not in kb]
    s = dict(unsupported=", ".join(unsupported), has_unsupported=int(len(unsupported) > 0),
             has_dose=int(bool(DOSE.search(answer))), words=len(answer.split()))
    if "virus" in disease.lower():
        s["says_no_cure"] = int(bool(NO_CURE.search(answer)))
    return s

print(len(eval_set), "questions |", score("Late_blight", "Use mancozeb 2 g per litre or chlorothalonil."))

20 questions | {'unsupported': 'chlorothalonil', 'has_unsupported': 1, 'has_dose': 1, 'words': 8}


In [9]:
# Cell 9 – run the SLM evaluation
rows = []
for name, rag in [("SLM+RAG", True), ("SLM no-RAG", False)]:
    for i, (d, q) in enumerate(eval_set, 1):
        ans, sec = ask_slm(build_prompt(d, 0.9, q, "English", rag), use_rag=rag)
        rows.append(dict(system=name, disease=d, question=q, seconds=round(sec, 2), answer=ans, **score(d, ans)))
        print(f"\r{name}: {i}/{len(eval_set)}", end="")
    print()
res = pd.DataFrame(rows)
res.to_csv(f"{PROJ}/results/advisor_eval_slm.csv", index=False)
res.groupby("system")[["has_unsupported", "has_dose", "words", "seconds"]].mean().round(2)

SLM+RAG: 20/20
SLM no-RAG: 20/20


,has_unsupported,has_dose,words,seconds
system,,,,
SLM no-RAG,0.45,0.25,185.95,75.18
SLM+RAG,0.00,0.00,19.70,17.17


In [10]:
# Cell 10 – language test (auto language)
LANG_Q = {"Hindi": "अब मुझे क्या करना चाहिए?", "Odia": "ଏବେ ମୁଁ କ'ଣ କରିବି?", "Tamil": "நான் இப்போது என்ன செய்ய வேண்டும்?",
          "Telugu": "నేను ఇప్పుడు ఏమి చేయాలి?", "Bengali": "আমি এখন কী করব?", "Marathi": "आता मी काय करावे?"}
LANG_DISEASES = ["Late_blight", "Tomato_Yellow_Leaf_Curl_Virus"]

def script_ratio(text, lang):
    lo, hi = SCRIPTS["Hindi" if lang == "Marathi" else lang]
    letters = [ch for ch in text if ch.isalpha()]
    return sum(lo <= ord(ch) <= hi for ch in letters) / max(len(letters), 1)

def run_language_test(ask, system_name):
    out = []
    for lang, q in LANG_Q.items():
        for d in LANG_DISEASES:
            ans, sec = ask(build_prompt(d, 0.9, q, "auto", True), use_rag=True)
            out.append(dict(system=system_name, disease=d, language=lang, seconds=round(sec, 2), answer=ans,
                            script_ratio=round(script_ratio(ans, lang), 2), **score(d, ans)))
    return out

lang_res = pd.DataFrame(run_language_test(ask_slm, "SLM+RAG"))
lang_res["correct_script"] = (lang_res.script_ratio >= 0.6).astype(int)
lang_res.to_csv(f"{PROJ}/results/advisor_eval_language.csv", index=False)
lang_res.pivot_table(index="language", columns="system", values="script_ratio").round(2)

system,SLM+RAG
language,
Bengali,0.0
Hindi,0.0
Marathi,0.0
Odia,0.0
Tamil,0.0
Telugu,0.0


In [11]:
# Cell 11 – optional Gemini comparison (needs GEMINI_API_KEY in Colab Secrets)
RUN_GEMINI = True
if RUN_GEMINI:
    !pip -q install google-genai
    from google import genai
    from google.genai import types
    from google.colab import userdata
    client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
    names = [m.name.replace("models/", "") for m in client.models.list()]
    LLM_MODEL = next((n for n in ["gemini-3.5-flash-lite", "gemini-3.5-flash", "gemini-flash-latest"] if n in names), "gemini-flash-latest")
    print("Gemini model:", LLM_MODEL)

    class QuotaDone(Exception): pass

    def ask_gemini(prompt, use_rag=True):
        t0 = time.time()
        for attempt in range(3):
            try:
                r = client.models.generate_content(model=LLM_MODEL, contents=prompt,
                        config=types.GenerateContentConfig(system_instruction=SYSTEM if use_rag else None,
                                                           temperature=0.2, max_output_tokens=4096))
                return (r.text or "").strip(), time.time() - t0
            except Exception as e:
                if "per day" in str(e).lower() or "PerDay" in str(e):
                    raise QuotaDone()                              # daily limit -> stop
                if "503" in str(e) or "429" in str(e):
                    time.sleep(15 * (attempt + 1))                 # busy / per-minute limit -> wait
                else:
                    raise
        return "", time.time() - t0

    g_rows = []
    try:
        for i, (d, q) in enumerate(eval_set, 1):
            ans, sec = ask_gemini(build_prompt(d, 0.9, q, "English", True))
            g_rows.append(dict(system="Gemini+RAG", disease=d, question=q, seconds=round(sec, 2), answer=ans, **score(d, ans)))
            print(f"\rGemini: {i}/{len(eval_set)}", end="")
            time.sleep(6)
    except QuotaDone:
        print(f"\nDaily Gemini limit reached after {len(g_rows)} answers – keeping those.")
    if g_rows:
        res = pd.concat([res[res.system != "Gemini+RAG"], pd.DataFrame(g_rows)], ignore_index=True)
        res.to_csv(f"{PROJ}/results/advisor_eval_slm.csv", index=False)
    print("\nGemini answers:", len(g_rows))

Gemini model: gemini-3.5-flash-lite
Gemini: 20/20
Gemini answers: 20


In [12]:
# Cell 12 – summary table + examples
summary = res.groupby("system").agg(answers=("answer", "size"), unsupported_chem_pct=("has_unsupported", "mean"),
                                    dose_given_pct=("has_dose", "mean"), avg_words=("words", "mean"),
                                    avg_seconds=("seconds", "mean"))
summary = summary.join(res[res.disease.str.contains("virus", case=False)]
                       .groupby("system")["says_no_cure"].mean().rename("virus_no_cure_pct"))
summary = summary.join(lang_res.groupby("system")["correct_script"].mean().rename("any_language_ok_pct"))
for c in summary.columns:
    if c.endswith("_pct"): summary[c] = (summary[c] * 100).round(1)
summary.insert(0, "model_size", [f"{n_params/1e9:.1f} B (local)" if s.startswith("SLM") else "cloud LLM"
                                 for s in summary.index])
summary = summary.round(2)
summary.to_csv(f"{PROJ}/results/advisor_eval_summary.csv")
display(summary)

ex = res[(res.disease == "Late_blight") & res.question.str.contains("how much")]
for _, r in ex.iterrows():
    print(f"##### {r.system} ({r.seconds}s) | unsupported: {r.unsupported or '-'} | dose: {r.has_dose}\n{r.answer}\n")
print(((res.has_unsupported == 1) | (res.has_dose == 1)).groupby(res.system).sum().rename("problem answers"))

,model_size,answers,unsupported_chem_pct,dose_given_pct,avg_words,avg_seconds,virus_no_cure_pct,any_language_ok_pct
system,,,,,,,,
Gemini+RAG,cloud LLM,20,0.0,0.0,58.60,1.48,25.0,NaN
SLM no-RAG,0.6 B (local),20,45.0,25.0,185.95,75.18,0.0,NaN
SLM+RAG,0.6 B (local),20,0.0,0.0,19.70,17.17,0.0,0.0


##### SLM+RAG (17.54s) | unsupported: - | dose: 0
- **Use**: Mancozeb  
- **How much**: Follow the product label instructions.

##### SLM no-RAG (65.84s) | unsupported: bacillus, sulfur | dose: 1
If your tomato leaf is affected by **Late Blight**, it's important to treat the plant with a **late blight-specific spray** that targets this particular disease.

### Recommended Chemical Spray:
- **Bacillus thuringianus (BT)**  
  - This is the most effective treatment for late blight in tomatoes.
  - It works by killing the bacteria on the leaves, which helps prevent spread.

### How Much to Use:
- **Apply BT at a rate of 1 mL per 500 g of soil or 2 mL per 1 L water**.
- Apply **every 7–14 days** if the plant is not already dead.

### Additional Tips:
- Keep the plants away from sources of infection (e.g., infected plants).
- Monitor the plant closely for signs of recovery or new infections.
- If the damage is severe, consider using a **combination spray** with other fungicides like **micona